# Resolver report

Data comes from `reports/resolver.duckdb`, refreshed by `scripts/report.sh` before this notebook opens.

Run all cells (**Kernel → Restart & Run All**) to see the current state of your benchmark runs.

In [ ]:
from pathlib import Path
import os
import duckdb


def _repo_root():
    here = Path(os.getcwd()).resolve()
    for p in [here, *here.parents]:
        if (p / "go.mod").exists():
            return p
    raise RuntimeError("repo root (go.mod) not found — run via scripts/report.sh")


ROOT = _repo_root()
DB = ROOT / "reports" / "resolver.duckdb"
if not DB.exists():
    raise RuntimeError(f"{DB} missing — run scripts/report.sh first")

con = duckdb.connect(str(DB), read_only=True)
print(f"connected: {DB}")

## What's in the database?

In [ ]:
con.sql("SHOW TABLES").to_df()

## Runs — one row per benchmark run

`virtual_model` is the harness-facing name (what you pass as `--model`); `real_model` is the backing model from the sidecar `run-config.yaml`. `overall` is PASS / PARTIAL / FAIL; `p95_ms` is the 95th-percentile per-query latency.

In [ ]:
con.sql("""
    SELECT run_id, model AS virtual_model, cfg_real_model AS real_model,
           overall, correct_count, query_count, p95_ms
    FROM run_summary
    ORDER BY run_id DESC
""").to_df()

### Real-model × scenario outcome matrix

Rows: `real-model (virtual)` so you can see each variant without losing the real-model grouping. Columns: scenario. Cells: most recent score.

In [ ]:
# Rows: real model (virtual variant) — prefer cfg_real_model from
# the sidecar run-config.yaml, fall back to resolved_real_model from the
# /v1/models probe, fall back to the virtual name. The "(virtual)" suffix
# preserves per-config detail when several virtuals map to the same real
# model (e.g. gresh-general + gresh-creative both backed by Qwen3.6).
df = con.sql("""
    SELECT COALESCE(
               NULLIF(cfg_real_model, ''),
               NULLIF(resolved_real_model, ''),
               model
           ) || ' (' || model || ')' AS model_label,
           scenario_id, score, run_id
    FROM comparison
    ORDER BY run_id DESC
""").to_df()

recent = df.drop_duplicates(subset=["model_label", "scenario_id"], keep="first")
recent.pivot(index="model_label", columns="scenario_id", values="score")

## Per-query results

Flat view: one row per (run, scenario). Limited to the most recent 50 rows; edit the SQL to widen.

In [ ]:
con.sql("""
    SELECT run_id, tier, model AS virtual_model,
           cfg_real_model AS real_model,
           scenario_id, score, elapsed_ms
    FROM comparison
    ORDER BY run_id DESC, tier, scenario_id
    LIMIT 50
""").to_df()

## Community benchmarks

Reference scores from the broader LLM ecosystem. `model_key` is the normalised form used to join against the real model name.

In [ ]:
con.sql("""
    SELECT model, model_key, benchmark, metric, value, as_of
    FROM community_benchmarks
    ORDER BY model, benchmark
""").to_df()

### Model × benchmark matrix

Pivoted view — rows are normalised `model_key`, columns are `benchmark/metric` pairs.

In [ ]:
# Pivot: one row per model_key (normalised name so quant / casing /
# org-prefix variants collapse), one column per benchmark/metric pair,
# values are the published reference scores.
df = con.sql("""
    SELECT model_key,
           benchmark || '/' || metric AS bench_metric,
           value
    FROM community_benchmarks
""").to_df()

df.pivot_table(index="model_key", columns="bench_metric", values="value", aggfunc="first")

## Next

- Drill into a single run in **`reproducibility.ipynb`**.
- Write your own SQL below — `con` is a read-only DuckDB connection and `.to_df()` renders a pandas DataFrame.
- Re-run `scripts/report.sh` after new benchmark runs to refresh.